# Fine-tune A2 trên Google Colab free

**Runtime → Change runtime type → GPU**. Nếu Colab cấp T4, notebook dùng NF4 + FP16. Chạy các cell theo thứ tự.

Chuẩn bị file `a2_colab_bundle.zip` đã được tạo trên máy. ZIP chứa dữ liệu, code và adapter A2 v3 khoảng 70 MB; model nền Qwen3-1.7B sẽ tải riêng từ Hugging Face.

Lượt này **tiếp tục từ A2 v3**, không tạo adapter ngẫu nhiên: 880 mẫu RAFT + 220 lượt ôn tập lấy từ 31 mẫu cũ đã kiểm tra, learning rate 3e-5, 2 epoch, batch hiệu dụng 8. Có 72 mẫu validation; không dùng 93 test.

Colab free không bảo đảm GPU/thời lượng phiên. Không dùng thủ thuật giữ phiên. Checkpoint lưu trên Drive mỗi 25 optimizer step; mất phiên có thể mất phần việc kể từ checkpoint hoàn chỉnh gần nhất.

In [ ]:
from google.colab import drive, files
from pathlib import Path, PurePosixPath
import hashlib, json, zipfile

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/DentalA2')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE = DRIVE_ROOT / 'a2_colab_bundle.zip'
if not BUNDLE.exists():
    print('Chọn a2_colab_bundle.zip từ máy tính (chỉ cần tải một lần).')
    uploaded = files.upload()
    if len(uploaded) != 1 or next(iter(uploaded)) != 'a2_colab_bundle.zip':
        raise ValueError('Cần chọn đúng một file a2_colab_bundle.zip')
    BUNDLE.write_bytes(uploaded['a2_colab_bundle.zip'])
    del uploaded

WORK = Path('/content/dental_a2')
WORK.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    names = z.namelist()
    if len(names) != len(set(names)):
        raise ValueError('ZIP có tên trùng')
    manifest = json.loads(z.read('bundle_manifest.json'))
    if set(names) != set(manifest['files_sha256']) | {'bundle_manifest.json'}:
        raise ValueError('ZIP không khớp manifest')
    for name in names:
        p = PurePosixPath(name)
        if p.is_absolute() or '..' in p.parts or '\\' in name:
            raise ValueError('Đường dẫn ZIP không hợp lệ')
    for name, digest in manifest['files_sha256'].items():
        if hashlib.sha256(z.read(name)).hexdigest() != digest:
            raise ValueError('Sai checksum: ' + name)
    z.extractall(WORK)
print('Đã kiểm tra và giải nén:', WORK)

In [ ]:
%pip install -q -r /content/dental_a2/requirements-colab-a2.txt

## Cấu hình phiên

Giữ nguyên `RUN_NAME` để chạy tiếp sau khi Colab ngắt. Nếu đổi dữ liệu/cấu hình cho thí nghiệm khác, dùng tên mới. Lệnh chạy bằng tiến trình Python riêng để nhận đúng thư viện vừa cài; không cần cài Modal.

Thư mục kết quả: `MyDrive/DentalA2/runs/<RUN_NAME>/`. Checkpoint có cả optimizer, scheduler và RNG để resume; adapter cuối cùng không đủ để khôi phục đầy đủ tiến trình train.

In [ ]:
import os, subprocess, sys
RUN_NAME = 'a2_v3_raft_v2_colab_01'
RUN_DIR = DRIVE_ROOT / 'runs' / RUN_NAME
env = dict(os.environ, PYTHONPATH=str(WORK), HF_HOME='/content/hf_cache_dental',
           TOKENIZERS_PARALLELISM='false', USE_TF='0', USE_FLAX='0')
def run_stage(stage):
    subprocess.run([sys.executable, '-u', '-X', 'utf8', '-m', 'scripts.train_a2_colab',
                    '--stage', stage, '--output-dir', str(RUN_DIR)], cwd=WORK, env=env, check=True)
subprocess.run(['nvidia-smi'], check=True)
print('Kết quả lưu tại:', RUN_DIR)

## 1. Kiểm tra dữ liệu và token — chưa tải trọng số model

In [ ]:
run_stage('preflight')

## 2. Đánh giá A2 v3 trước khi train

72 câu, prompt yêu cầu quote như lượt sau train, cùng NF4/FP16 và greedy decoding. Không so trực tiếp các tỷ lệ này với lượt Modal BF16 cũ. Chạy lại cell sẽ tiếp tục các câu chưa có kết quả.

In [ ]:
run_stage('before')

## 3. Huấn luyện tiếp từ A2 v3

Nếu bị ngắt: kết nối lại GPU, chạy lại các cell thiết lập phía trên, giữ `RUN_NAME`, rồi chạy lại cell này. Chương trình chỉ nhận checkpoint có dấu hoàn tất và checksum đúng. Khôi phục từ checkpoint gần nhất, không phải từ từng batch cuối cùng.

Checkpoint nằm trực tiếp trên Drive, giữ tối đa 2 checkpoint theo cơ chế Trainer. Nếu đổi GPU/thư viện sau khi ngắt, tính toán có thể không giống tuyệt đối dù trạng thái optimizer và bộ sinh số được khôi phục.

In [ ]:
run_stage('train')

## 4. Đánh giá sau train và tạo 4 hình PNG

Nạp mới nền Qwen3 + adapter được chọn theo validation loss, cùng cấu hình FP16 với lượt trước train. Các chỉ số tự động đo schema, status, mã dẫn và quote; vẫn cần đọc nguồn để chấm đúng nội dung.

Các hình: `loss.png`, `before_after.png`, `04_confusion_matrix.png`, `05_f1_by_decision.png`. Before là **A2 v3**, After là **adapter mới**.

In [ ]:
run_stage('after')

## 5. Đóng gói và tải kết quả — checkpoint đầy đủ vẫn ở Drive

In [ ]:
run_stage('export')
RESULT_ZIP = RUN_DIR.parent / (RUN_NAME + '_results.zip')
print(RESULT_ZIP)
files.download(str(RESULT_ZIP))

Nguồn tham khảo: [Giới hạn Colab](https://research.google.com/colaboratory/faq.html), [PEFT và lượng tử hóa](https://huggingface.co/docs/peft/developer_guides/quantization), [Trainer resume](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Trainer.train). Notebook chưa được chạy trên một phiên Colab thật khi được chuẩn bị.